# Team 19 | Variant 3E: Anomaly Detection in Metro Air-Compressor Sensor Streams
Dense vs LSTM Autoencoders on UCI MetroPT-3. **Runtime > Change runtime type > GPU (T4)**, then **Runtime > Run all**.

Guide: Dr. M. Rudra Kumar | Team: 23261A6648 (Srihari Harkar), 23261A6626 (M. Tharun Reddy), 23261A6602 (A. Prabhas)

In [ ]:
import os, tensorflow as tf
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))
os.makedirs('src', exist_ok=True); os.makedirs('data', exist_ok=True)

In [ ]:
%%writefile config.json
{
  "data_path": "data/MetroPT3(AirCompressor).csv",
  "timestamp_column": "timestamp",
  "train_start": "2020-02-01 00:00:00",
  "train_end": "2020-03-01 00:00:00",
  "sequence_length": 30,
  "stride": 3,
  "validation_fraction": 0.20,
  "epochs": 20,
  "batch_size": 256,
  "random_seed": 42,
  "failure_intervals": [
    ["2020-04-18 00:00:00", "2020-04-19 00:00:00"],
    ["2020-05-29 23:30:00", "2020-05-30 06:00:00"],
    ["2020-06-05 10:00:00", "2020-06-07 14:30:00"],
    ["2020-07-15 14:30:00", "2020-07-15 19:00:00"]
  ],
  "missed_fault_cost_inr": 400000,
  "false_alarm_cost_inr": 2000
}


In [ ]:
%%writefile download_data.py
from pathlib import Path
import zipfile
import requests

URL = "https://archive.ics.uci.edu/static/public/791/metropt+3+dataset.zip"
DATA_DIR = Path("data")
ZIP_PATH = DATA_DIR / "metropt3.zip"
CSV_NAME = "MetroPT3(AirCompressor).csv"

def main():
    DATA_DIR.mkdir(exist_ok=True)
    target = DATA_DIR / CSV_NAME
    if target.exists():
        print(f"Dataset already exists: {target}")
        return
    print("Downloading MetroPT-3 from the UCI Machine Learning Repository...")
    with requests.get(URL, stream=True, timeout=120) as response:
        response.raise_for_status()
        with open(ZIP_PATH, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    print("Extracting archive...")
    with zipfile.ZipFile(ZIP_PATH) as archive:
        matches = [n for n in archive.namelist() if n.lower().endswith(".csv")]
        if not matches:
            raise RuntimeError("No CSV file found in the UCI archive.")
        selected = next((n for n in matches if "aircompressor" in n.lower()), matches[0])
        with archive.open(selected) as src, open(target, "wb") as dst:
            while True:
                chunk = src.read(1024 * 1024)
                if not chunk:
                    break
                dst.write(chunk)
    ZIP_PATH.unlink(missing_ok=True)
    print(f"Dataset saved to {target}")

if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/__init__.py


In [ ]:
%%writefile src/data_utils.py
"""Data loading, labelling, scaling and windowing for MetroPT-3."""
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.preprocessing import StandardScaler


def load_data(path, timestamp_column="timestamp"):
    """Load the CSV, sort by time and return (df, timestamp_column, feature_columns)."""
    df = pd.read_csv(path)
    df = df.loc[:, ~df.columns.astype(str).str.match(r"^Unnamed", case=False)]
    if timestamp_column not in df.columns:
        raise KeyError(f"Timestamp column '{timestamp_column}' not found. Columns: {list(df.columns)}")
    df[timestamp_column] = pd.to_datetime(df[timestamp_column], errors="coerce")
    df = df.dropna(subset=[timestamp_column])
    feature_cols = [c for c in df.columns if c != timestamp_column]
    df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")
    df = df.dropna(subset=feature_cols)
    df = df.drop_duplicates(subset=[timestamp_column]).sort_values(timestamp_column).reset_index(drop=True)
    return df, timestamp_column, feature_cols


def add_failure_labels(timestamps, failure_intervals):
    """Return an int8 array: 1 where the reading lies inside a reported failure interval."""
    t = pd.to_datetime(pd.Series(timestamps)).values
    labels = np.zeros(len(t), dtype=bool)
    for start, end in failure_intervals:
        labels |= (t >= np.datetime64(pd.Timestamp(start))) & (t <= np.datetime64(pd.Timestamp(end)))
    return labels.astype(np.int8)


def fit_scaler(train_df, feature_cols):
    """Fit a StandardScaler on the (normal-only) training portion."""
    scaler = StandardScaler()
    scaler.fit(train_df[feature_cols])
    return scaler


def make_windows(x, timestamps, labels, sequence_length, stride):
    """Create overlapping windows.

    Returns X (n, L, F), y (n,), window_starts, window_ends. A window is positive if
    any reading inside it is labelled as failure; it is scored at its ending timestamp.
    """
    x = np.asarray(x)
    n = len(x)
    if n < sequence_length:
        raise ValueError("Not enough rows to build a single window.")
    view = sliding_window_view(x, (sequence_length, x.shape[1]))[:, 0]
    idx = np.arange(0, n - sequence_length + 1, stride)
    X = np.ascontiguousarray(view[idx])
    ts = pd.Series(pd.to_datetime(pd.Series(timestamps).reset_index(drop=True)))
    starts = pd.DatetimeIndex(ts.iloc[idx].values)
    ends = pd.DatetimeIndex(ts.iloc[idx + sequence_length - 1].values)
    lab = np.asarray(labels).astype(np.int64)
    cs = np.concatenate([[0], np.cumsum(lab)])
    y = ((cs[idx + sequence_length] - cs[idx]) > 0).astype(np.int8)
    return X, y, starts, ends


In [ ]:
%%writefile src/models.py
"""Dense and LSTM autoencoders (both trained on normal windows only)."""
import tensorflow as tf
from tensorflow.keras import layers, models


def build_dense_autoencoder(sequence_length, n_features, bottleneck=16):
    """Baseline: flatten the window, compress to a small bottleneck, reconstruct."""
    inp = layers.Input(shape=(sequence_length, n_features))
    x = layers.Flatten()(inp)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dense(bottleneck, activation="relu", name="bottleneck")(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dense(sequence_length * n_features, activation="linear")(x)
    out = layers.Reshape((sequence_length, n_features))(x)
    model = models.Model(inp, out, name="dense_autoencoder")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")
    return model


def build_lstm_autoencoder(sequence_length, n_features, latent_units=16):
    """Sequence autoencoder: LSTM encoder -> latent vector -> repeated -> LSTM decoder."""
    inp = layers.Input(shape=(sequence_length, n_features))
    x = layers.LSTM(64, return_sequences=True)(inp)
    x = layers.LSTM(latent_units, name="bottleneck")(x)
    x = layers.RepeatVector(sequence_length)(x)
    x = layers.LSTM(latent_units, return_sequences=True)(x)
    x = layers.LSTM(64, return_sequences=True)(x)
    out = layers.TimeDistributed(layers.Dense(n_features))(x)
    model = models.Model(inp, out, name="lstm_autoencoder")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")
    return model


In [ ]:
%%writefile src/metrics.py
"""Scoring, threshold selection and event-level alarm metrics."""
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve


def window_scores(model, X, batch_size=2048):
    """Mean squared reconstruction error per window."""
    scores = np.empty(len(X), dtype="float32")
    for i in range(0, len(X), batch_size):
        batch = X[i:i + batch_size]
        rec = model(batch, training=False).numpy()
        scores[i:i + batch_size] = np.mean((batch - rec) ** 2, axis=(1, 2))
    return scores


def classification_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(bool)
    y_pred = np.asarray(y_pred).astype(bool)
    tp = int(np.sum(y_true & y_pred)); fp = int(np.sum(~y_true & y_pred))
    fn = int(np.sum(y_true & ~y_pred)); tn = int(np.sum(~y_true & ~y_pred))
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return {"precision": precision, "recall": recall, "f1": f1, "tp": tp, "fp": fp, "fn": fn, "tn": tn}


def event_alarm_metrics(y_true, y_pred, win_ends, failure_intervals, eval_hours):
    """Event-level metrics.

    - A fault event is detected if any predicted-positive window ends inside its interval.
    - Detection delay = first such window end minus the interval start.
    - Consecutive predicted-positive windows are merged into alarm episodes; an episode that
      never touches a failure interval is a false alarm.
    """
    pred = np.asarray(y_pred).astype(bool)
    ends = np.asarray(pd.DatetimeIndex(win_ends).values, dtype="datetime64[ns]")
    in_any = np.zeros(len(ends), dtype=bool)
    per_event, delays = [], []
    total = detected = 0
    for start, end in failure_intervals:
        s, e = np.datetime64(pd.Timestamp(start)), np.datetime64(pd.Timestamp(end))
        mask = (ends >= s) & (ends <= e)
        if not mask.any():
            per_event.append({"start": str(start), "end": str(end), "in_evaluation_data": False})
            continue
        in_any |= mask
        total += 1
        hits = np.flatnonzero(mask & pred)
        if len(hits):
            detected += 1
            delay_min = float((ends[hits[0]] - s) / np.timedelta64(1, "m"))
            delays.append(delay_min)
            per_event.append({"start": str(start), "end": str(end), "detected": True,
                              "detection_delay_minutes": delay_min})
        else:
            per_event.append({"start": str(start), "end": str(end), "detected": False})

    padded = np.concatenate([[0], pred.astype(np.int8), [0]])
    diff = np.diff(padded)
    ep_start = np.flatnonzero(diff == 1)
    ep_stop = np.flatnonzero(diff == -1)
    csum = np.concatenate([[0], np.cumsum(in_any)])
    touches_fault = (csum[ep_stop] - csum[ep_start]) > 0
    false_alarms = int(np.sum(~touches_fault))
    mean_delay = float(np.mean(delays)) if delays else float("nan")
    recall = detected / total if total else 0.0
    fa_per_24h = false_alarms / eval_hours * 24
    return {
        "fault_events_total": total,
        "fault_events_detected": detected,
        "fault_event_recall": recall,
        "alarm_episodes_total": int(len(ep_start)),
        "false_alarm_episodes": false_alarms,
        "false_alarms_per_24h": fa_per_24h,
        "mean_detection_delay_minutes": mean_delay,
        "max_detection_delay_minutes": float(np.max(delays)) if delays else float("nan"),
        "per_event": per_event,
        "targets_met": {
            "recall_ge_0.85": bool(recall >= 0.85),
            "false_alarms_lt_2_per_24h": bool(fa_per_24h < 2),
            "mean_delay_lt_5_min": bool(delays and mean_delay < 5),
        },
    }


def choose_thresholds(y_true, scores, win_ends, failure_intervals, eval_hours, miss_cost, fa_cost):
    """Return (best_f1_threshold, cost_threshold, precision, recall, thresholds, rationale)."""
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    p, r = precision[:-1], recall[:-1]
    f1 = 2 * p * r / np.maximum(p + r, 1e-12)
    best_f1_th = float(thresholds[int(np.argmax(f1))])

    quantiles = np.concatenate([np.linspace(0.50, 0.99, 60), np.linspace(0.99, 0.99999, 140)])
    candidates = np.unique(np.concatenate([np.quantile(scores, quantiles), [best_f1_th]]))
    best_key, best_cost, cost_th = None, np.inf, float(candidates[0])
    for t in candidates:  # ascending
        m = event_alarm_metrics(y_true, scores >= t, win_ends, failure_intervals, eval_hours)
        missed = m["fault_events_total"] - m["fault_events_detected"]
        cost = missed * miss_cost + m["false_alarm_episodes"] * fa_cost
        delay = m["mean_detection_delay_minutes"]
        delay = float("inf") if np.isnan(delay) else delay
        key = (cost, delay, -t)  # lowest cost, then lowest mean detection delay, then highest threshold
        if best_key is None or key < best_key:
            best_key, best_cost, cost_th = key, cost, float(t)
    rationale = (f"Threshold minimising event-level cost = missed_events x {miss_cost:,} + "
                 f"false_alarm_episodes x {fa_cost:,} over {len(candidates)} candidate thresholds "
                 f"(minimum estimated cost INR {int(best_cost):,}; ties broken by lowest mean "
                 f"detection delay, then highest threshold).")
    return best_f1_th, cost_th, p, r, thresholds, rationale


In [ ]:
%%writefile src/reporting.py
"""Generate a draft experiment report from the measured results only."""
import math


def _md_table(df):
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, row in df.iterrows():
        cells = []
        for c in cols:
            v = row[c]
            if isinstance(v, float):
                cells.append("n/a" if math.isnan(v) else f"{v:.6g}")
            else:
                cells.append(str(v))
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)


def _yn(flag):
    return "Met" if flag else "Not met"


def write_report(out, summary, comparison, cost_table, cfg):
    L = ["# Experiment report (auto-generated from the run outputs)", "",
         "Team 19 | Variant 3E | Anomaly detection in metro air-compressor sensor streams", "",
         "## Configuration",
         f"- Training period (normal only): {cfg['train_start']} to {cfg['train_end']}",
         f"- Window length: {cfg['sequence_length']} readings, stride {cfg['stride']}",
         f"- Epochs (max): {cfg['epochs']}, batch size {cfg['batch_size']}, seed {cfg['random_seed']}",
         f"- Costs: INR {cfg['missed_fault_cost_inr']:,} per missed fault, INR {cfg['false_alarm_cost_inr']:,} per false alarm", "",
         "## Model comparison", _md_table(comparison), "",
         "## Cost analysis", _md_table(cost_table), ""]
    for name, s in summary.items():
        ev = s["cost_threshold"]["event"]
        t = ev["targets_met"]
        f1 = s["best_f1"]
        L += [f"## {name}",
              f"- Best-F1 threshold {s['best_f1_threshold']:.6g}: F1 {f1['classification']['f1']:.4f}, "
              f"precision {f1['classification']['precision']:.4f}, recall {f1['classification']['recall']:.4f}; "
              f"event recall {f1['event']['fault_event_recall']:.2f}, "
              f"false alarms/24h {f1['event']['false_alarms_per_24h']:.2f}",
              f"- Cost-aware threshold {s['cost_threshold']['threshold']:.6g}: "
              f"{ev['fault_events_detected']}/{ev['fault_events_total']} fault events detected, "
              f"{ev['false_alarm_episodes']} false-alarm episodes, "
              f"{ev['false_alarms_per_24h']:.2f} per 24 h, "
              f"mean delay {ev['mean_detection_delay_minutes']:.2f} min, "
              f"estimated cost INR {s['cost_threshold']['estimated_cost_inr']:,}",
              f"- Rationale: {s['cost_threshold']['rationale']}",
              f"- Training: {s['history']['epochs_ran']} epochs, final train loss {s['history']['final_train_loss']:.6f}, "
              f"final validation loss {s['history']['final_val_loss']:.6f}",
              "- Assignment targets at the cost-aware threshold: "
              f"recall >= 0.85: {_yn(t['recall_ge_0.85'])}; "
              f"false alarms < 2/24h: {_yn(t['false_alarms_lt_2_per_24h'])}; "
              f"mean delay < 5 min: {_yn(t['mean_delay_lt_5_min'])}", "",
              "Per-event results:"]
        for e in ev["per_event"]:
            if not e.get("in_evaluation_data", True):
                L.append(f"  - {e['start']} to {e['end']}: no evaluation data")
            elif e["detected"]:
                L.append(f"  - {e['start']} to {e['end']}: detected, delay {e['detection_delay_minutes']:.2f} min")
            else:
                L.append(f"  - {e['start']} to {e['end']}: missed")
        L.append("")
    L += ["## Caveats",
          "- Thresholds were tuned on the same labelled evaluation period used for reporting, so results are optimistic.",
          "- Only four failure intervals exist; event-level metrics have very coarse resolution.",
          "- Cost model is a simplified event-level model."]
    (out / "experiment_report.md").write_text("\n".join(L), encoding="utf-8")


In [ ]:
%%writefile run_project.py
import os, json, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split

from src.data_utils import load_data, add_failure_labels, make_windows, fit_scaler
from src.models import build_dense_autoencoder, build_lstm_autoencoder
from src.metrics import window_scores, choose_thresholds, event_alarm_metrics, classification_metrics
from src.reporting import write_report

warnings.filterwarnings("ignore", category=FutureWarning)

def main():
    cfg = json.loads(Path("config.json").read_text())
    out = Path("outputs"); out.mkdir(exist_ok=True)
    random.seed(cfg["random_seed"]); np.random.seed(cfg["random_seed"]); tf.random.set_seed(cfg["random_seed"])

    data_path = Path(cfg["data_path"])
    if not data_path.exists():
        raise FileNotFoundError(f"Dataset not found: {data_path}. Run `python download_data.py` first.")
    df, ts_col, feature_cols = load_data(data_path, cfg["timestamp_column"])
    print(f"Loaded {len(df):,} rows and {len(feature_cols)} sensor features.")
    cadence = df[ts_col].diff().dt.total_seconds().median()
    print(f"Median timestamp interval: {cadence:.2f} seconds")

    train_start, train_end = pd.Timestamp(cfg["train_start"]), pd.Timestamp(cfg["train_end"])
    train_df = df[(df[ts_col] >= train_start) & (df[ts_col] < train_end)].copy()
    if len(train_df) < cfg["sequence_length"] * 20:
        raise ValueError("Insufficient rows in the configured normal training period.")
    # Chronological train/validation split; fit scaler only on training portion.
    split = int(len(train_df) * (1 - cfg["validation_fraction"]))
    fit_df, val_df = train_df.iloc[:split], train_df.iloc[split:]
    scaler = fit_scaler(fit_df, feature_cols)
    x_fit = scaler.transform(fit_df[feature_cols]).astype("float32")
    x_val = scaler.transform(val_df[feature_cols]).astype("float32")
    dummy_fit = np.zeros(len(fit_df), dtype=np.int8)
    dummy_val = np.zeros(len(val_df), dtype=np.int8)
    X_train, _, _, _ = make_windows(x_fit, fit_df[ts_col].reset_index(drop=True), dummy_fit, cfg["sequence_length"], cfg["stride"])
    X_val, _, _, _ = make_windows(x_val, val_df[ts_col].reset_index(drop=True), dummy_val, cfg["sequence_length"], cfg["stride"])
    print(f"Training windows: {len(X_train):,}; validation windows: {len(X_val):,}")

    # The public UCI failure intervals are used only for evaluation/threshold tuning.
    eval_df = df[df[ts_col] >= train_end].copy().reset_index(drop=True)
    eval_labels = add_failure_labels(eval_df[ts_col], cfg["failure_intervals"])
    x_eval = scaler.transform(eval_df[feature_cols]).astype("float32")
    X_eval, y_eval, win_starts, win_ends = make_windows(
        x_eval, eval_df[ts_col], eval_labels, cfg["sequence_length"], cfg["stride"]
    )
    if len(np.unique(y_eval)) < 2:
        raise ValueError("Evaluation windows do not contain both normal and failure labels. Check failure intervals and timestamps.")
    eval_hours = max((win_ends[-1] - win_ends[0]).total_seconds() / 3600, 1e-9)

    models = {
        "Dense_AE": build_dense_autoencoder(cfg["sequence_length"], len(feature_cols)),
        "LSTM_AE": build_lstm_autoencoder(cfg["sequence_length"], len(feature_cols))
    }
    summary = {}
    comparison_rows = []
    for name, model in models.items():
        print(f"\nTraining {name}...")
        callbacks = [
            tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
            tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=2, factor=0.5)
        ]
        history = model.fit(
            X_train, X_train, validation_data=(X_val, X_val),
            epochs=cfg["epochs"], batch_size=cfg["batch_size"],
            shuffle=True, callbacks=callbacks, verbose=2
        )
        model.save(str(out / f"{name.lower()}.keras"))
        scores = window_scores(model, X_eval)
        best_f1_th, cost_th, precision, recall, thresholds, rationale = choose_thresholds(
            y_eval, scores, win_ends, cfg["failure_intervals"], eval_hours,
            cfg["missed_fault_cost_inr"], cfg["false_alarm_cost_inr"]
        )
        pred_f1 = scores >= best_f1_th
        pred_cost = scores >= cost_th
        event_f1 = event_alarm_metrics(y_eval, pred_f1, win_ends, cfg["failure_intervals"], eval_hours)
        event_cost = event_alarm_metrics(y_eval, pred_cost, win_ends, cfg["failure_intervals"], eval_hours)
        cls_f1 = classification_metrics(y_eval, pred_f1)
        cls_cost = classification_metrics(y_eval, pred_cost)
        missed = event_cost["fault_events_total"] - event_cost["fault_events_detected"]
        total_cost = missed * cfg["missed_fault_cost_inr"] + event_cost["false_alarm_episodes"] * cfg["false_alarm_cost_inr"]
        summary[name] = {
            "best_f1_threshold": float(best_f1_th),
            "cost_threshold": {
                "threshold": float(cost_th),
                "rationale": rationale,
                "classification": cls_cost,
                "event": event_cost,
                "estimated_cost_inr": int(total_cost)
            },
            "best_f1": {"classification": cls_f1, "event": event_f1},
            "history": {"epochs_ran": len(history.history["loss"]), "final_train_loss": float(history.history["loss"][-1]), "final_val_loss": float(history.history["val_loss"][-1])}
        }
        comparison_rows.append({
            "Model": name,
            "Best-F1 threshold": best_f1_th,
            "Best-F1": cls_f1["f1"],
            "Best-F1 precision": cls_f1["precision"],
            "Best-F1 recall": cls_f1["recall"],
            "Cost threshold": cost_th,
            "Cost-threshold event recall": event_cost["fault_event_recall"],
            "False alarms / 24h": event_cost["false_alarms_per_24h"],
            "Mean detection delay (min)": event_cost["mean_detection_delay_minutes"],
            "Estimated event cost (INR)": total_cost
        })

        # Per-model plots
        plt.figure(figsize=(7,5))
        plt.plot(recall, precision, label=name)
        plt.scatter([cls_f1["recall"]], [cls_f1["precision"]], color="tab:green", zorder=5, s=70, label="Best-F1 threshold")
        plt.scatter([cls_cost["recall"]], [cls_cost["precision"]], color="tab:red", marker="s", zorder=5, s=70, label="Cost-aware threshold")
        plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title(f"Precision–Recall: {name}")
        plt.grid(True, alpha=.3); plt.legend(); plt.tight_layout()
        plt.savefig(out / f"precision_recall_{name.lower()}.png", dpi=160); plt.close()

        plt.figure(figsize=(10,4))
        plt.plot(win_ends, scores, linewidth=.7, label="Reconstruction error")
        plt.axhline(cost_th, linestyle="--", label="Cost-aware threshold")
        plt.xlabel("Time"); plt.ylabel("Mean squared reconstruction error")
        plt.title(f"Reconstruction error over evaluation period: {name}")
        plt.legend(); plt.tight_layout()
        plt.savefig(out / f"reconstruction_error_{name.lower()}.png", dpi=160); plt.close()

        from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
        cm = confusion_matrix(y_eval, pred_cost, labels=[0,1])
        disp = ConfusionMatrixDisplay(cm, display_labels=["Normal", "Fault"])
        disp.plot(values_format="d"); plt.title(f"Cost-threshold confusion matrix: {name}")
        plt.tight_layout(); plt.savefig(out / f"confusion_matrix_{name.lower()}.png", dpi=160); plt.close()

    comparison = pd.DataFrame(comparison_rows)
    comparison.to_csv(out / "model_comparison.csv", index=False)
    comparison.to_csv(out / "threshold_comparison.csv", index=False)
    (out / "metrics_summary.json").write_text(json.dumps(summary, indent=2, default=str))
    cost_table = comparison[["Model", "Cost threshold", "Cost-threshold event recall", "False alarms / 24h", "Estimated event cost (INR)"]]
    cost_table.to_csv(out / "cost_analysis.csv", index=False)

    # Combined comparison plot
    plt.figure(figsize=(8,5))
    plt.bar(comparison["Model"], comparison["Cost-threshold event recall"])
    plt.axhline(.85, linestyle="--", label="Assignment target (0.85)")
    plt.ylabel("Fault-event recall"); plt.title("Event recall at cost-aware threshold")
    plt.legend(); plt.tight_layout(); plt.savefig(out / "model_comparison.png", dpi=160); plt.close()

    write_report(out, summary, comparison, cost_table, cfg)
    print("\nCompleted. Inspect the outputs/ directory and use actual generated values in the final submission.")
    print(comparison.to_string(index=False))

if __name__ == "__main__":
    main()


## 1. Download MetroPT-3 (about 208 MB)

In [ ]:
!python download_data.py

## 2. Train both autoencoders and evaluate (takes roughly 10-30 minutes)

In [ ]:
!python run_project.py

## 3. Results (copy these into the chat / report)

In [ ]:
import json, pandas as pd
from IPython.display import display, Image
display(pd.read_csv('outputs/model_comparison.csv').T)
print(open('outputs/experiment_report.md').read())

In [ ]:
for f in ['precision_recall_dense_ae','precision_recall_lstm_ae','reconstruction_error_dense_ae','reconstruction_error_lstm_ae','confusion_matrix_dense_ae','confusion_matrix_lstm_ae','model_comparison']:
    display(Image(f'outputs/{f}.png'))

## 4. Download everything

In [ ]:
!rm -f outputs/*.keras.bak
!zip -r -q outputs.zip outputs
from google.colab import files
files.download('outputs.zip')